# Малюємо точки на Google Maps

Малюємо розвіддані з РЕБ-чатику

In [1]:
from pathlib import Path
import json

base = Path("/Users/kb/workspace/wildducks-intel/wdintel")
with open('files.json', 'r') as f: input_files_verbose = json.load(f)
for x in input_files_verbose:
    x["path"] = base / x["path"]
all_input_files = [x["path"] for x in input_files_verbose]

In [2]:
# library stuff

# convert 452043N 0323130E coords to (45.45277777777778, 32.6) via the
# `decimal degrees = degrees + (minutes/60 + seconds/60)` formula
# example: ne_to_pair(45, 20, 43, 32, 31, 30)
#
# see https://stackoverflow.com/questions/61429200/how-do-i-convert-these-coordinates-to-real-numbers
#
# n: float
# e: float
# returns: (float, float)
def ne_to_pair(n1, n2, n3, e1, e2, e3):
    x = float(n1) + float(n2) / 60 + float(n3) / 360
    y = float(e1) + float(e2) / 60 + float(e3) / 630
    return (x, y)

import mgrs

# example mgrs_str: "37TDP1004003060". see https://medium.com/dida-machine-learning/understanding-and-converting-mgrs-coordinates-in-python-5a3ee87d75ff
def mgrs_to_latlon(mgrs_str):
    m = mgrs.MGRS()
    return m.toLatLon(mgrs_str)


## Збір

Спочатку зберемо файли. Перетворюємо docx -> txt наступним чином:

```
/Applications/LibreOffice.app/Contents/MacOS/soffice --headless --convert-to "txt:Text (encoded):UTF8" input.docx
```


In [3]:
import subprocess

force_libreoffice_reprocess = False


# # : [{path: Path, txt: String}]
# txt_results = []
def gen_txt_results(input_files):
    txt_results = []
    for path in input_files:
        if path.suffix in [".pdf"]:
            txt_results.append({"path": path, "txt": ""})
            continue
        path_txt = path.with_suffix(".txt")
        # print(f"""input path: {path}\nname: {path.name}\nstem: {path.stem}\npath_txt: {path_txt}\n""")
        if path_txt.exists() and not force_libreoffice_reprocess:
            pass
            # print("> path_txt exists. skipping since force_libreoffice_reprocess is False")
        else:
            print("> running soffice subprocess...")
            args = [
                "/Applications/LibreOffice.app/Contents/MacOS/soffice",
                "--headless",
                "--convert-to",
                "txt:Text (encoded):UTF8",
                path
            ]
            print(f"> args: {args}")
            res = subprocess.run(args, cwd=path.parent, check=True, capture_output=True)
            print(f"""> done. returncode: {res.returncode}\n> stderr: {res.stderr}\n> stdout: {res.stdout}""")
        txt_result = ""
        with open(path_txt, 'r') as f:
            txt_result = f.read()
        # print(f"> read {len(txt_result)} symbols")
        txt_results.append({"path": path, "txt": txt_result})
    return txt_results
_txt_results = gen_txt_results(all_input_files)

> running soffice subprocess...
> args: ['/Applications/LibreOffice.app/Contents/MacOS/soffice', '--headless', '--convert-to', 'txt:Text (encoded):UTF8', PosixPath('/Users/kb/workspace/wildducks-intel/wdintel/3. 26.08.22_1715.docx')]
> done. returncode: 0
> stderr: b''
> stdout: b'convert /Users/kb/workspace/wildducks-intel/wdintel/3. 26.08.22_1715.docx -> /Users/kb/workspace/wildducks-intel/wdintel/3. 26.08.22_1715.txt using filter : Text (encoded):UTF8\n'


In [4]:
import re
# 36T XT 84075 00749
p_NE = re.compile(r'(?P<N_coord>[0-9][0-9][0-9][0-9][0-9]+)N,?\s*(?P<E_coord>[0-9][0-9][0-9][0-9][0-9][0-9]+)E')
p_LAT_LNG = re.compile(r'(?P<lat_coord>[0-9]+\.[0-9]+),?\s*\s+(?P<lng_coord>[0-9]+\.[0-9]+)')
p_MGRS_with_spaces = re.compile(r'(?P<MGRS>[0-9][0-9][A-Z] ?[A-Z][A-Z] ?[0-9]+ ?[0-9]+)')
p_emptyLine = re.compile(r'^[\s|\*]*$')

# # : [{x: real, y: real, label: String, info: String, fname: String}]
# coord_results = []
def gen_coord_results(txt_results):
    coord_results = []
    for res in txt_results:
        # print(f"> res.path: {res['path']}")
        symbol_counter = 0
        paragraph_begin = 0
        paragraph_end = 0
        for line2 in res["txt"][paragraph_begin:].splitlines():
            paragraph_end += len(line2)
            if p_emptyLine.search(line2) is not None:
                break
        for line in res["txt"].splitlines():
            symbol_counter += len(line) + 1
            # print(f"> line: {line}")
            found_one = False
            r_NE = p_NE.search(line)
            if r_NE is not None:
                found_one = True
            r_MGRS = p_MGRS_with_spaces.search(line)
            if r_MGRS is not None:
                found_one = True
            r_LAT_LNG = p_LAT_LNG.search(line)
            if r_LAT_LNG is not None:
                found_one = True

            # initialize to fill later
            label = ""
            info = ""
            x = None
            y = None

            if found_one:
                label = line[:40]
                info = res["txt"][paragraph_begin:paragraph_end]
                info = info.replace(line, f"<strong>{line}</strong>")

            if r_NE is not None:
                n_s = r_NE.groupdict()["N_coord"]
                e_s = r_NE.groupdict()["E_coord"]
                # print(f"> found NE: lat: {n_s}; lng: {e_s}")
                info = info.replace(n_s, f"<strong>{n_s}</strong>")
                info = info.replace(e_s, f"<strong>{e_s}</strong>")
                x, y = ne_to_pair(
                    float(n_s[:2]), float(n_s[2:4]), float(n_s[4:]),
                    float(e_s[:3]), float(e_s[3:5]), float(e_s[5:]),
                )

            if r_MGRS is not None:
                mgrs_val_orig = r_MGRS.groupdict()["MGRS"]
                mgrs_val = mgrs_val_orig.replace(" ", "")
                info = info.replace(mgrs_val_orig, f"<strong>{mgrs_val_orig}</strong>")
                # print(f"> found MGRS: {mgrs_val}")
                # print(f"> calling mgrs_val with mgrs_val: {mgrs_val}")
                # print(f"> latlon: {mgrs_to_latlon(mgrs_val)}")
                x, y = mgrs_to_latlon(mgrs_val)

            if r_LAT_LNG is not None:
                lat_coord_s = r_LAT_LNG.groupdict()["lat_coord"]
                lng_coord_s = r_LAT_LNG.groupdict()["lng_coord"]
                info = info.replace(lat_coord_s, f"<strong>{lat_coord_s}</strong>")
                info = info.replace(lng_coord_s, f"<strong>{lng_coord_s}</strong>")
                # print(f"> found r_LAT_LNG: {lat_coord_s}, {lng_coord_s}")
                x = float(lat_coord_s)
                y = float(lng_coord_s)

            info = f"<strong>file name: {res['path'].name}</strong><br><br>{info}"
            info = info.replace("\n", "<br><br>\n")

            if found_one:
                coord_results.append({
                    "x": x,
                    "y": y,
                    "label": label, 
                    "info": info,
                    "fname": "{}".format(res["path"].name),
                })
                # print(f"> found: [{res['path'].name}] x: {x}, y: {y}, n_s: {n_s}, e_s: {e_s}\n> label: {label}\n> info: {info}\n")
                # print(f"> found: [{res['path'].name}] x: {x}, y: {y}, n_s: {n_s}, e_s: {e_s}\n> label: {label}\n")

            if p_emptyLine.search(line) is not None:
                paragraph_begin = symbol_counter
                # print(f"> new paragraph_begin is at: {res['txt'][paragraph_begin:paragraph_begin+20]}")
                paragraph_end = paragraph_begin
                for line2 in res["txt"][paragraph_begin:].splitlines():
                    paragraph_end += len(line2)
                    if p_emptyLine.search(line2) is not None:
                        break
    return coord_results
_coord_results = gen_coord_results(gen_txt_results(all_input_files))


## Малюємо

In [5]:
import gmplot
import json
from IPython.display import IFrame
import urllib.parse

apikey = ''
with open('GOOGLE_MAPS_KEY', 'r') as f:
    apikey = f.read()
# print("> apikey: {}".format(apikey))

def draw(input_files_verbose, outfile):
    input_files = [x["path"] for x in input_files_verbose]
    all_fnames = [x.name for x in input_files]
    coord_results = gen_coord_results(gen_txt_results(input_files))
    used_files_list_s = ""
    for fv in input_files_verbose:
        context_or_empty = ""
        if len(fv["context"]) > 0:
            context_or_empty = f' – {fv["context"]}'
        map_link = f'<a target=_blank href="/wdintel/map-{fv["path"].with_suffix(".html").name}">map</a>'
        used_files_list_s += f'<li><a href="/wdintel/{urllib.parse.quote(fv["path"].name)}" target=_blank>{fv["path"].name}</a> – {map_link}{context_or_empty}</li>\n'

    with open('map.tpl', 'r') as f: map_tpl = f.read()

    # TODO: gen filename
    with open(outfile, 'w') as f:
        f.write(map_tpl\
                .replace("{apikey}", apikey)\
                .replace("{coord_results}", json.dumps(coord_results))\
                .replace("{used_files_list_s}", used_files_list_s))

# draw for every file
for fv in input_files_verbose:
    outfile = f'wdintel/map-{fv["path"].with_suffix(".html").name}'
    draw([fv], outfile)
# draw for all files
draw(input_files_verbose, 'wdintel/map.html')

IFrame(src='./wdintel/map.html', width=1000, height=600)